## just to get a bit of the grasp of the data in the dataset

In [1]:
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
import numpy as np

In [2]:
envs = ["atari_battle_zone", "atari_double_dunk", "atari_phoenix", "atari_this_game", "atari_battle_zone", "box2d_lunar_lander", "box2d_continuous_lunar_lander", "box2d_bipedal_walker", "cc_acrobot", "cc_cartpole", "cc_mountain_car", "cc_continuous_mountain_car", "cc_pendulum", "minigrid_door_key", "minigrid_empty_random", "minigrid_four_rooms", "minigrid_unlock", "brax_ant", "brax_halfcheetah", "brax_hopper", "brax_humanoid"]
algos = ["ppo", "dqn", "sac"]

data = []
for env in envs:
    for algo in algos:
        try:
            partial_data = pd.read_csv(f"arlbench_data/256_10/{env}_{algo}.csv")
            partial_data["env_name"] = env
            partial_data["algorithm"] = algo
            data.append(partial_data)
        except FileNotFoundError:
            continue
data = pd.concat(data)

In [ ]:
# normalization cross-algorithm per environment
data["normed_performance"] = data.groupby("env_name")["performance"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-8)
)

# on data_two we normalize per env and algorithm
data["normed_performance_per_alg"] = data.groupby(["env_name", "algorithm"])["performance"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-8)
)

# on data_two we normalize per env and algorithm
data["_seed_avg"] = data.groupby(["env_name", "algorithm", "config_id"])["performance"].transform("mean")
data["normed_performance_per_alg_seedavg"] = data.groupby(["env_name"], group_keys=False)["_seed_avg"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-8)
)
# data.drop(columns=["_seed_avg"], inplace=True)

# data["performance_seed_avg"]

In [23]:
# [x for x in data.columns if not x.startswith("hp")]

In [25]:
data_slim = data[["config_id", "env_name", "algorithm", "seed", "performance", "last_performance", "_seed_avg", "last_performance_unnormalized", "max_performance", "normed_performance", "normed_performance_per_alg", "normed_performance_per_alg_seedavg"]].copy()
# sort by config id
data_slim = data_slim.sort_values(by=["config_id", "env_name"])
data_slim.head(11)

,config_id,env_name,algorithm,seed,performance,last_performance,_seed_avg,last_performance_unnormalized,max_performance,normed_performance,normed_performance_per_alg,normed_performance_per_alg_seedavg
0,0,atari_battle_zone,ppo,9,4742.1875,0.111053,2750.78125,3397.65625,0.192605,0.121790,0.121790,0.152163
256,0,atari_battle_zone,ppo,0,3726.5625,0.111053,2750.78125,3397.65625,0.192605,0.095706,0.095706,0.152163
512,0,atari_battle_zone,ppo,7,0.0000,0.111053,2750.78125,3397.65625,0.192605,0.000000,0.000000,0.152163
768,0,atari_battle_zone,ppo,6,3054.6875,0.111053,2750.78125,3397.65625,0.192605,0.078451,0.078451,0.152163
1024,0,atari_battle_zone,ppo,1,8351.5625,0.111053,2750.78125,3397.65625,0.192605,0.214486,0.214486,0.152163
1280,0,atari_battle_zone,ppo,8,1000.0000,0.111053,2750.78125,3397.65625,0.192605,0.025682,0.025682,0.152163
1536,0,atari_battle_zone,ppo,4,2250.0000,0.111053,2750.78125,3397.65625,0.192605,0.057785,0.057785,0.152163
1792,0,atari_battle_zone,ppo,3,210.9375,0.111053,2750.78125,3397.65625,0.192605,0.005417,0.005417,0.152163
2048,0,atari_battle_zone,ppo,2,757.8125,0.111053,2750.78125,3397.65625,0.192605,0.019462,0.019462,0.152163
2304,0,atari_battle_zone,ppo,5,3414.0625,0.111053,2750.78125,3397.65625,0.192605,0.087681,0.087681,0.152163


In [5]:
## performance: per seed performance
## lats_performance: avg over seed performance at the end of training
## last_performance_unnormalized: same as last_performance but without normalization
## max_performance: average over seeds (how?) max performance achieved during training (presumably)

In [6]:
data_slim["performance_normalized"] = data_slim["performance"] / data_slim["last_performance_unnormalized"] * data_slim["last_performance"]
data_slim.head(10)

,config_id,env_name,algorithm,seed,performance,last_performance,last_performance_unnormalized,max_performance,performance_normalized
0,0,atari_battle_zone,ppo,9,4742.1875,0.111053,3397.65625,0.192605,0.154999
256,0,atari_battle_zone,ppo,0,3726.5625,0.111053,3397.65625,0.192605,0.121803
512,0,atari_battle_zone,ppo,7,0.0000,0.111053,3397.65625,0.192605,0.000000
768,0,atari_battle_zone,ppo,6,3054.6875,0.111053,3397.65625,0.192605,0.099843
1024,0,atari_battle_zone,ppo,1,8351.5625,0.111053,3397.65625,0.192605,0.272972
1280,0,atari_battle_zone,ppo,8,1000.0000,0.111053,3397.65625,0.192605,0.032685
1536,0,atari_battle_zone,ppo,4,2250.0000,0.111053,3397.65625,0.192605,0.073542
1792,0,atari_battle_zone,ppo,3,210.9375,0.111053,3397.65625,0.192605,0.006895
2048,0,atari_battle_zone,ppo,2,757.8125,0.111053,3397.65625,0.192605,0.024769
2304,0,atari_battle_zone,ppo,5,3414.0625,0.111053,3397.65625,0.192605,0.111589


In [ ]:
# How are the algorithms normalized? Is this a fair normalization?
# It appears that the normalization is done for each (algorithm, environment) pair separately, 
# 

from sklearn.linear_model import LinearRegression

# normalization coefficients 
# {"{alg}-{env}": (coef, intercept)} 
# where normalized = coef * unnormalized + intercept

normalization_coefficients = {}

for env in data_slim["env_name"].unique():
    for alg in data_slim["algorithm"].unique():

        data_subset = data_slim[(data_slim["algorithm"] == alg) & (data_slim["env_name"] == env)]

        try:
            X = data_subset[["last_performance_unnormalized"]]
            y = data_subset["last_performance"]
            reg = LinearRegression().fit(X, y)
            normalization_coefficients[f"{alg}-{env}"] = (reg.coef_[0], reg.intercept_)
        except ValueError:
            # print(f"Not enough data for {env} and {alg}")
            continue
        

atari_battle_zone - ppo: R^2=1.0000, Coef=0.000035281, Intercept=-0.0088
atari_battle_zone - dqn: R^2=1.0000, Coef=0.000043676, Intercept=-0.0077
atari_double_dunk - ppo: R^2=1.0000, Coef=0.059325176, Intercept=1.0832
atari_double_dunk - dqn: R^2=1.0000, Coef=0.083289953, Intercept=1.9863
atari_phoenix - ppo: R^2=1.0000, Coef=0.000083388, Intercept=-0.0015
atari_phoenix - dqn: R^2=1.0000, Coef=0.000110683, Intercept=-0.0021
atari_this_game - ppo: R^2=1.0000, Coef=0.000107038, Intercept=-0.0225
atari_this_game - dqn: R^2=1.0000, Coef=0.000097149, Intercept=-0.0124
box2d_bipedal_walker - ppo: R^2=1.0000, Coef=0.002548897, Intercept=0.5098
box2d_bipedal_walker - sac: R^2=1.0000, Coef=0.002398910, Intercept=0.3139
box2d_continuous_lunar_lander - ppo: R^2=1.0000, Coef=0.002189157, Intercept=0.4378
box2d_continuous_lunar_lander - sac: R^2=1.0000, Coef=0.002068341, Intercept=0.4137
box2d_lunar_lander - ppo: R^2=1.0000, Coef=0.002390375, Intercept=0.4781
box2d_lunar_lander - dqn: R^2=1.0000, C

In [13]:
## Alternatively: Normalize Cross algorithms in a single environment

normalization_coefficients_cross_alg = {}

for env in data_slim["env_name"].unique():

    env_data = data_slim[(data_slim["env_name"] == env)]

    max_performance = env_data["performance"].max()
    min_performance = env_data["performance"].min()

    coef = 1 / (max_performance - min_performance) if max_performance != min_performance else 1.0
    intercept = -min_performance * coef

    normalization_coefficients_cross_alg[env] = (coef, intercept)

    print(f"Env: {env}, Coef: {coef:.9f}, Intercept: {intercept:.9f}") 

Env: atari_battle_zone, Coef: 0.000025682, Intercept: -0.000000000
Env: atari_double_dunk, Coef: 0.044016506, Intercept: 1.056396149
Env: atari_phoenix, Coef: 0.000099535, Intercept: -0.001695205
Env: atari_this_game, Coef: 0.000150945, Intercept: -0.000000000
Env: box2d_bipedal_walker, Coef: 0.002533674, Intercept: 0.506734810
Env: box2d_continuous_lunar_lander, Coef: 0.002178022, Intercept: 0.435604386
Env: box2d_lunar_lander, Coef: 0.002894084, Intercept: 0.578816704
Env: brax_ant, Coef: 0.000195234, Intercept: 0.390467189
Env: brax_halfcheetah, Coef: 0.000099893, Intercept: 0.199786049
Env: brax_hopper, Coef: 0.000176210, Intercept: 0.352419952
Env: brax_humanoid, Coef: 0.000107825, Intercept: 0.215649361
Env: cc_acrobot, Coef: 0.002366951, Intercept: 1.183475720
Env: cc_cartpole, Coef: 0.002035882, Intercept: -0.017941214
Env: cc_continuous_mountain_car, Coef: 0.005026233, Intercept: 0.502120717
Env: cc_mountain_car, Coef: 0.011012647, Intercept: 2.202529407
Env: cc_pendulum, Coef